# receive

> Connect to the Swift Transcribe app (or any `nadakit` BLE sender) and type received text into the focused app.


In [ ]:
#| default_exp receive


In [ ]:
#| hide
from nbdev.showdoc import *


Run this on the PC while the Swift Transcribe app has **Send to PC over Bluetooth** on:

```sh
pixi run receive
# or: python -m nadakit.receive
```

Logs go to stderr with timestamps (`[nadakit-recv]`). Use `--verbose` for scan advertisements too.


In [ ]:
#| export
import argparse, asyncio, logging, sys, time, traceback


In [ ]:
#| export
log = logging.getLogger('nadakit.receive')

def _setup_log(verbose=False):
    "Configure stderr logging once."
    if log.handlers: return
    h = logging.StreamHandler(sys.stderr)
    h.setFormatter(logging.Formatter('[%(asctime)s] [nadakit-recv] %(levelname)s %(message)s', '%H:%M:%S'))
    log.addHandler(h)
    log.setLevel(logging.DEBUG if verbose else logging.INFO)
    log.propagate = False


In [ ]:
#| export
try:
    from bleak import BleakClient, BleakScanner
    _bleak = True
except Exception:
    _bleak = False

try:
    from .bt import SERVICE, CHAR, NAME
except Exception:
    SERVICE = '6e616461-6b69-7400-4b42-000000000001'
    CHAR = '6e616461-6b69-7400-4b42-000000000002'
    NAME = 'nadakit'


In [ ]:
#| export
CODES_WIN = dict(space=0x20, **{'return':0x0D}, tab=0x09, backspace=0x08, escape=0x1B,
                 left=0x25, up=0x26, right=0x27, down=0x28)
CODES_XD = dict(space='space', **{'return':'Return'}, tab='Tab', backspace='BackSpace', escape='Escape',
                left='Left', right='Right', up='Up', down='Down')


In [ ]:
#| export
def _type_mac(k):
    from . import keys
    if k in keys.CODES: return keys.type_key(k)
    ok = True
    for ch in k:
        ok = keys.type_key(ch) and ok
    return ok

def _type_win(k):
    import ctypes
    from ctypes import wintypes
    user32 = ctypes.windll.user32
    INPUT_KEYBOARD, KEYEVENTF_KEYUP, KEYEVENTF_UNICODE = 1, 0x0002, 0x0004
    ULONG_PTR = ctypes.c_size_t
    class KEYBDINPUT(ctypes.Structure):
        _fields_ = [('wVk', wintypes.WORD), ('wScan', wintypes.WORD), ('dwFlags', wintypes.DWORD),
                    ('time', wintypes.DWORD), ('dwExtraInfo', ULONG_PTR)]
    class MOUSEINPUT(ctypes.Structure):
        _fields_ = [('dx', wintypes.LONG), ('dy', wintypes.LONG), ('mouseData', wintypes.DWORD),
                    ('dwFlags', wintypes.DWORD), ('time', wintypes.DWORD), ('dwExtraInfo', ULONG_PTR)]
    class HARDWAREINPUT(ctypes.Structure):
        _fields_ = [('uMsg', wintypes.DWORD), ('wParamL', wintypes.WORD), ('wParamH', wintypes.WORD)]
    class INPUTUNION(ctypes.Union):
        _fields_ = [('ki', KEYBDINPUT), ('mi', MOUSEINPUT), ('hi', HARDWAREINPUT)]
    class INPUT(ctypes.Structure):
        _fields_ = [('type', wintypes.DWORD), ('u', INPUTUNION)]
    def tap(vk=0, scan=0, flags=0):
        i = INPUT(type=INPUT_KEYBOARD, u=INPUTUNION(ki=KEYBDINPUT(vk, scan, flags, 0, 0)))
        user32.SendInput(1, ctypes.byref(i), ctypes.sizeof(i))
    if k in CODES_WIN:
        vk = CODES_WIN[k]; tap(vk=vk); tap(vk=vk, flags=KEYEVENTF_KEYUP); return True
    for ch in k:
        data = ch.encode('utf-16-le')
        for i in range(0, len(data), 2):
            scan = int.from_bytes(data[i:i+2], 'little')
            tap(scan=scan, flags=KEYEVENTF_UNICODE)
            tap(scan=scan, flags=KEYEVENTF_UNICODE | KEYEVENTF_KEYUP)
    return True

def _type_linux(k):
    import shutil, subprocess
    if not shutil.which('xdotool'): return False
    if k in CODES_XD: subprocess.run(['xdotool', 'key', CODES_XD[k]], check=False); return True
    subprocess.run(['xdotool', 'type', '--clearmodifiers', '--', k], check=False); return True


In [ ]:
#| export
def inject(k):
    "Type `k` into the focused app on this machine. False when the platform won't allow it."
    t0 = time.perf_counter()
    try:
        if sys.platform == 'darwin': ok = _type_mac(k)
        elif sys.platform == 'win32': ok = _type_win(k)
        elif sys.platform.startswith('linux'): ok = _type_linux(k)
        else: ok = False
    except Exception:
        log.exception('inject raised for %r', k)
        return False
    ms = (time.perf_counter() - t0) * 1000
    log.info('inject %s %r (%d chars) in %.1fms platform=%s', 'ok' if ok else 'FAIL', k[:80], len(k), ms, sys.platform)
    return ok


In [ ]:
#| export
def _match(d, ad, nearby=None):
    name = (d.name or ad.local_name or '').lower()
    uuids = {u.lower() for u in (ad.service_uuids or [])}
    hit = name == NAME or SERVICE.lower() in uuids
    if nearby is not None and not hit:
        label = d.name or ad.local_name
        if label: nearby.add(label)
    if hit:
        log.info('scan HIT name=%r local=%r addr=%s rssi=%s uuids=%s', d.name, ad.local_name, d.address, getattr(ad, 'rssi', None), list(uuids)[:4])
    else:
        # Only chatter about unnamed noise at DEBUG; named devices at most once per scan via `nearby`.
        log.debug('scan see name=%r local=%r addr=%s rssi=%s uuids=%s', d.name, ad.local_name, d.address, getattr(ad, 'rssi', None), list(uuids)[:2])
    return hit

async def find(timeout=20.0):
    "Scan for a nadakit keyboard peripheral."
    if not _bleak: raise RuntimeError('Install bleak: pip install bleak')
    log.info('scanning for name=%r service=%s timeout=%.1fs', NAME, SERVICE, timeout)
    t0 = time.perf_counter()
    nearby = set()
    def filt(d, ad): return _match(d, ad, nearby)
    dev = await BleakScanner.find_device_by_filter(filt, timeout=timeout)
    elapsed = time.perf_counter() - t0
    if dev:
        log.info('scan done in %.1fs found=%s', elapsed, getattr(dev, 'address', None))
    else:
        phones = sorted(n for n in nearby if 'iphone' in n.lower() or 'ipad' in n.lower())
        others = sorted(nearby - set(phones))[:6]
        if phones:
            log.warning('scan done in %.1fs — saw %s but it is NOT advertising as %r. On the phone: open Transcribe, keep it foreground, tap Link, wait until status says Advertising.',
                        elapsed, ', '.join(phones), NAME)
        elif nearby:
            log.warning('scan done in %.1fs — no %r. Nearby named devices: %s', elapsed, NAME, ', '.join(others))
        else:
            log.warning('scan done in %.1fs — no %r advertisement (and no named BLE devices). Keep Transcribe open with Link on.', elapsed, NAME)
    return dev


In [ ]:
#| export
def resolve_char(client):
    "Pick our notify characteristic by service UUID, then handle (avoids Bleak duplicate-UUID errors)."
    svc_l, char_l = SERVICE.lower(), CHAR.lower()
    matches = []
    for svc in client.services or []:
        for ch in svc.characteristics:
            if ch.uuid.lower() != char_l: continue
            matches.append((svc.uuid.lower() == svc_l, ch))
            log.info('gatt candidate service=%s char=%s handle=%s props=%s ourservice=%s',
                     svc.uuid, ch.uuid, getattr(ch, 'handle', None), ch.properties, svc.uuid.lower() == svc_l)
    if not matches:
        log.error('no characteristic %s found', CHAR); return None
    # Prefer characteristic that lives under our service UUID.
    for ours, ch in matches:
        if ours and 'notify' in (ch.properties or []): return ch
    for ours, ch in matches:
        if ours: return ch
    for ours, ch in matches:
        if 'notify' in (ch.properties or []): return ch
    return matches[0][1]


In [ ]:
#| export
async def run(timeout=20.0):
    "Connect, subscribe, and type each notified key until interrupted."
    if not _bleak: raise SystemExit('Install bleak first: pip install bleak')
    n_connect, n_notify, n_disconnect = 0, 0, 0
    while True:
        dev = await find(timeout=timeout)
        if not dev:
            log.warning('no peripheral found — open Transcribe and enable “Send to PC over Bluetooth” / Link')
            await asyncio.sleep(2); continue
        n_connect += 1
        log.info('connecting #%d to name=%r addr=%s', n_connect, dev.name, dev.address)
        disconnected = asyncio.Event()
        def on_disconnect(client):
            nonlocal n_disconnect
            n_disconnect += 1
            log.warning('disconnect callback #%d connected=%s', n_disconnect, client.is_connected)
            disconnected.set()
        try:
            async with BleakClient(dev, disconnected_callback=on_disconnect, timeout=20.0) as client:
                svcs = list(client.services) if client.services else []
                log.info('connected mtu=%s services=%d', getattr(client, 'mtu_size', '?'), len(svcs))
                ch = resolve_char(client)
                if ch is None:
                    raise RuntimeError(f'characteristic {CHAR} not found under service {SERVICE}')
                handle = getattr(ch, 'handle', None)
                log.info('using char handle=%s uuid=%s props=%s', handle, ch.uuid, ch.properties)
                def on_notify(sender, data: bytearray):
                    nonlocal n_notify
                    n_notify += 1
                    text = data.decode('utf-8', errors='replace')
                    log.info('notify #%d sender=%s bytes=%d hex=%s text=%r', n_notify, sender, len(data), data[:24].hex(), text[:120])
                    inject(text)
                # Prefer handle to avoid “Multiple Characteristics with this UUID”.
                target = handle if handle is not None else ch
                log.info('start_notify target=%r', target)
                await client.start_notify(target, on_notify)
                log.info('subscribed — focus a text field on this PC, then speak in Transcribe')
                while client.is_connected and not disconnected.is_set():
                    await asyncio.sleep(0.5)
                log.warning('left connection loop is_connected=%s', client.is_connected)
        except asyncio.CancelledError:
            log.info('cancelled'); raise
        except Exception as e:
            n_disconnect += 1
            log.error('connection error #%d: %s: %s', n_disconnect, type(e).__name__, e)
            log.debug('traceback:\n%s', traceback.format_exc())
            await asyncio.sleep(1)
        log.info('totals connect=%d notify=%d disconnect=%d — reconnecting', n_connect, n_notify, n_disconnect)
        await asyncio.sleep(0.5)


In [ ]:
#| export
def main(argv=None):
    "CLI entry: `nadakit-receive`."
    p = argparse.ArgumentParser(description='Receive nadakit keys over Bluetooth LE and type them.')
    p.add_argument('--timeout', type=float, default=20.0, help='Seconds to scan before retrying')
    p.add_argument('-v', '--verbose', action='store_true', help='Log every advertisement seen while scanning')
    args = p.parse_args(argv)
    _setup_log(verbose=args.verbose)
    log.info('nadakit-receive starting platform=%s bleak=%s service=%s', sys.platform, _bleak, SERVICE)
    try: asyncio.run(run(timeout=args.timeout))
    except KeyboardInterrupt: log.info('stopped by user')


In [ ]:
#| export
if __name__ == '__main__': main()
